In [ ]:
import torch
import pandas as pd
import numpy as np
from collections import Counter

In [8]:
df = pd.read_csv("../data/fake_behavior.csv")
df.tail(2)

,t_dat,customer_id,article_id,event_type
2123499,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,582480024,click
2123500,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,751288001,click


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2123501 entries, 0 to 2123500
Data columns (total 4 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   t_dat        object
 1   customer_id  object
 2   article_id   int64 
 3   event_type   object
dtypes: int64(1), object(3)
memory usage: 64.8+ MB


In [ ]:
def build_graph(df, session_gap_days=7):
    df = df.copy()
    df["t_dat"] = pd.to_datetime(df["t_dat"])

    edges = []
    count = 0
    for user, user_df in df.groupby("customer_id"):
        user_df = user_df.sort_values("t_dat")

        items = user_df["article_id"].tolist()
        times = user_df["t_dat"].tolist()

        if len(items) < 2:
            continue

        session = [items[0]]
        print(f"user: {user[:8]}, {len(user_df)}")
        print(session)
        count+=1
        if count==5:
            break
        for i in range(1, len(items)):
            gap = (times[i] - times[i-1]).days

            if gap <= session_gap_days:
                session.append(items[i])
            else:
                edges += build_session_edges(session)
                session = [items[i]]
        edges += build_session_edges(session)

    # 🔥 COUNT WEIGHT
    edge_counter = Counter(edges)
    print(f"edge counter: {edge_counter}")

    all_items = set([a for (a, b) in edge_counter.keys()] + 
                    [b for (a, b) in edge_counter.keys()])

    id2idx = {aid: i for i, aid in enumerate(all_items)}

    edge_index = []
    edge_weight = []

    for (a, b), w in edge_counter.items():
        edge_index.append([id2idx[a], id2idx[b]])
        edge_weight.append(w)

    edge_index = torch.tensor(edge_index, dtype=torch.long).t()
    edge_weight = torch.tensor(edge_weight, dtype=torch.float)

    print("Num nodes:", len(id2idx))
    print("Num edges:", edge_index.shape[1])

    return edge_index, edge_weight, id2idx


def build_session_edges(session):
    edges = []
    for i in range(len(session)):
        for j in range(i + 1, len(session)):
            a, b = session[i], session[j]
            edges.append((a, b))
            edges.append((b, a))
    return edges

In [6]:
transactions_path = "../data/transactions_train.csv"
df = pd.read_csv(transactions_path)
df["article_id"] = df["article_id"].astype(str).str.zfill(10)
df.head(5)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2020-08-09,0004df3751c43c51f79aa24d321f35ce4a1a3ec180c4a1...,0832482001,0.016932,1
1,2020-08-09,0004df3751c43c51f79aa24d321f35ce4a1a3ec180c4a1...,0802881001,0.067780,1
2,2020-08-09,0004df3751c43c51f79aa24d321f35ce4a1a3ec180c4a1...,0783707047,0.005068,1
3,2020-08-09,000fb6e772c5d0023892065e659963da90b1866035558e...,0852585002,0.042356,2
4,2020-08-09,000fb6e772c5d0023892065e659963da90b1866035558e...,0852585002,0.042356,2


In [21]:
build_graph(df)

user: 00006413, 4
['0896152002']
user: 00009d94, 4
['0884319008']
user: 0001d44d, 8
['0900157002']
user: 00025f82, 3
['0781613016']
user: 0003e867, 3
['0927358001']
edge counter: Counter({('0900157002', '0900157002'): 2, ('0900157002', '0850244001'): 2, ('0850244001', '0900157002'): 2, ('0900157002', '0850244002'): 2, ('0850244002', '0900157002'): 2, ('0900157002', '0850244003'): 2, ('0850244003', '0900157002'): 2, ('0900157002', '0820484002'): 2, ('0820484002', '0900157002'): 2, ('0900157002', '0820484004'): 2, ('0820484004', '0900157002'): 2, ('0900157002', '0812818003'): 2, ('0812818003', '0900157002'): 2, ('0896152002', '0730683050'): 1, ('0730683050', '0896152002'): 1, ('0896152002', '0927530004'): 1, ('0927530004', '0896152002'): 1, ('0896152002', '0791587015'): 1, ('0791587015', '0896152002'): 1, ('0730683050', '0927530004'): 1, ('0927530004', '0730683050'): 1, ('0730683050', '0791587015'): 1, ('0791587015', '0730683050'): 1, ('0927530004', '0791587015'): 1, ('0791587015', '0927

In [10]:
filtered_df = df[df["t_dat"] == "2020-08-14"]
print(f"Số dòng với t_dat = 2020-08-14: {len(filtered_df)}")
filtered_df.head(10)

Số dòng với t_dat = 2020-08-14: 184758


,t_dat,customer_id,article_id,event_type
49,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,884319008,purchase
50,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,858052007,click
51,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,921226001,purchase
52,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,832282001,click
53,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,706016001,purchase
54,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,836364002,click
55,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,881244001,purchase
56,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,766402020,click
133,2020-08-14,00025f8226be50dcab09402a2cacd520a99e112fe01fdd...,781613016,purchase
134,2020-08-14,00025f8226be50dcab09402a2cacd520a99e112fe01fdd...,823791002,click


In [29]:
hist_df = df[df["t_dat"] != "2020-08-14"]
print(f"Số dòng với t_dat: {len(hist_df)}")
hist_df.head(5)
hist_df.to_csv("../data/hist_data.csv", index=False)

Số dòng với t_dat: 1938743


In [11]:
filtered_df.to_csv("../data/test.csv", index=False)

In [24]:
clip_data = torch.load("../data/clip_embeddings.pt")
data = torch.load("../data/article_embeddings_hgnn3.pt")

In [25]:
emb = data["embeddings"]
id2idx = data["id2idx"]

In [22]:
# In ra keys và 1 mẫu
print("Số keys:", len(clip_data.keys()))
print("\nCác keys (10 mẫu đầu):")
for i, key in enumerate(list(clip_data.keys())[:10]):
    print(f"  {key}")

# In ra 1 mẫu value
print("\n1 mẫu key-value:")
sample_key = list(clip_data.keys())[0]
print(f"Key: {sample_key}")
print(f"Value type: {type(clip_data[sample_key])}")
print(f"Value shape: {clip_data[sample_key].shape if hasattr(clip_data[sample_key], 'shape') else 'N/A'}")
print(f"Value: {clip_data[sample_key]}")

Số keys: 19284

Các keys (10 mẫu đầu):
  0108775044
  0111565001
  0111586001
  0111593001
  0111609001
  0120129001
  0120129014
  0123173001
  0126589007
  0126589010

1 mẫu key-value:
Key: 0108775044
Value type: <class 'torch.Tensor'>
Value shape: torch.Size([512])
Value: tensor([ 4.7273e-02,  6.4621e-02, -1.2524e-02,  2.1046e-02,  2.2154e-03,
        -3.0189e-02, -2.2084e-02, -5.6870e-03,  3.4435e-02, -1.1891e-02,
        -2.8226e-02, -6.4788e-02, -3.5684e-02, -2.2390e-03,  2.1674e-02,
         1.3855e-02, -7.1584e-02, -2.0553e-02, -2.9776e-03, -2.2854e-02,
        -1.4183e-02, -2.2618e-02, -3.4741e-02,  3.4026e-03, -2.5158e-02,
        -4.1193e-02, -1.3493e-03,  5.8652e-02, -4.0569e-02,  2.8711e-02,
        -8.1253e-02, -1.1579e-03,  3.0341e-02,  2.9245e-04,  5.1088e-03,
         1.9867e-02, -4.7859e-02,  1.3294e-02,  3.7160e-02, -1.0182e-02,
        -1.4246e-02, -4.5982e-02,  2.5034e-02,  2.5450e-02, -5.4300e-02,
        -4.3354e-03, -4.3745e-02,  8.7598e-03, -9.7880e-03, -6.5527

ModuleNotFoundError: No module named 'hgnn_model'

In [26]:
count =0 
for aid, idx in id2idx.items():
    print(f"aid: {aid}, idx: {idx}, clip[aid]: {clip_data[aid]}")
    count+=1
    if count==5: 
        break

aid: 0804691001, idx: 0, clip[aid]: tensor([-2.7453e-03,  2.1458e-02, -1.1507e-03,  4.8934e-02,  5.0829e-02,
        -8.8418e-02, -5.5898e-02,  2.7377e-02,  6.0033e-03,  5.5205e-02,
        -2.3292e-02,  1.3512e-02, -1.5547e-02, -4.0378e-02,  4.1804e-02,
         1.5855e-03, -1.3557e-01, -4.6019e-03,  7.8842e-03,  1.1295e-02,
        -2.8956e-02, -2.6889e-02, -5.8524e-03,  4.6037e-02, -7.6074e-03,
        -7.5044e-02,  3.2624e-02,  3.1794e-02, -3.4085e-02,  1.6035e-02,
         4.4593e-03,  7.0416e-02,  4.3510e-02,  2.2921e-02, -2.2443e-02,
         2.6625e-03, -2.8765e-02,  5.0812e-02, -2.8906e-02, -2.9266e-02,
        -2.7010e-02, -2.9344e-02,  6.1212e-02,  2.0099e-02, -2.9916e-02,
        -2.2835e-02, -5.9972e-02,  6.6185e-02, -7.6287e-02, -2.3912e-02,
        -1.0622e-02, -7.0510e-02, -6.2842e-03, -6.1542e-03, -1.8099e-02,
         4.3694e-02,  4.5749e-03,  8.9964e-03, -5.5017e-02,  3.3349e-02,
        -5.4911e-02, -4.6642e-02,  5.7501e-02,  7.9351e-02,  3.4872e-02,
         4.3267

In [28]:
data_test = pd.read_csv("../data/test.csv")
data_test

,t_dat,customer_id,article_id,event_type
0,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,884319008,purchase
1,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,858052007,click
2,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,921226001,purchase
3,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,832282001,click
4,2020-08-14,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,706016001,purchase
...,...,...,...,...
184753,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,742916003,click
184754,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,761696002,click
184755,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,796600003,click
184756,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,582480024,click
